# 04 — Feature Windows (`make_windows`)

M4 task 2. Build per-category sliding input/target windows with
`make_windows`, show the results, and record conclusions.

- Core logic: `src/demandlens/features/windows.py`
- Batch export: `scripts/build_windows.py`
- This notebook calls the same core function and re-checks the leakage contract.


In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("project root:", PROJECT_ROOT)


In [ ]:
import re

def load_window_config(path):
    text = path.read_text(encoding="utf-8-sig")
    task_block = re.search(r"\ntask:\n(.*?)(?=\n\w)", text, re.S).group(1)
    return {
        "lookback_days": [int(v) for v in re.findall(r"^\s*-\s*(\d+)\s*$", task_block, re.M)],
        "horizon_days": int(re.search(r"horizon_days:\s*(\d+)", task_block).group(1)),
        "step_days": int(re.search(r"step_days:\s*(\d+)", task_block).group(1)),
        "processed_dir": re.search(r"processed_dir:\s*(\S+)", text).group(1),
        "variant_output": re.search(r"'10':\s*\n.*?output:\s*(\S+)", text, re.S).group(1),
    }

cfg = load_window_config(PROJECT_ROOT / "config" / "config.yaml")
cfg


In [ ]:
from demandlens.data.quality import read_daily

daily_rows = read_daily(PROJECT_ROOT / cfg["processed_dir"] / cfg["variant_output"])
print("daily rows:", len(daily_rows))
print("first row:", daily_rows[0])


In [ ]:
from demandlens.features.windows import WINDOW_FIELDS, make_windows

results = {}
for lookback in cfg["lookback_days"]:
    windows, meta = make_windows(daily_rows, lookback, cfg["horizon_days"], cfg["step_days"])
    results[lookback] = (windows, meta)
    print(f"lookback={lookback}: {meta['n_windows']} windows / {meta['n_categories']} categories")
    print(f"  input span  {meta['input_start']} .. {meta['input_end']}")
    print(f"  target span {meta['target_start']} .. {meta['target_end']}")
    print(f"  first: {windows[0]}")


In [ ]:
from datetime import date, timedelta

def iso(d):
    return date.fromisoformat(d)

failures = []
for lookback, (windows, meta) in results.items():
    horizon = meta["horizon"]
    for w in windows:
        a, b, c, d = (iso(x) for x in (w["input_start_date"], w["input_end_date"], w["target_start_date"], w["target_end_date"]))
        if not (a <= b < c <= d):
            failures.append((w["sample_id"], "ordering"))
        if c != b + timedelta(days=1):
            failures.append((w["sample_id"], "target not adjacent to input"))
        if (b - a).days + 1 != lookback:
            failures.append((w["sample_id"], "input length != lookback"))
        if (d - c).days + 1 != horizon:
            failures.append((w["sample_id"], "target length != horizon"))

print("leakage/continuity failures:", failures)
assert not failures, failures
print("OK: target_start == input_end + 1 day; input length == lookback; target length == horizon")


## Conclusions

- `make_windows` slides per `category_id` independently; categories never share dates.
- `y` is the scalar sum of `purchase` over the 7 target days only.
- `target_start_date == input_end_date + 1 day` holds for every window, and the
  input/target lengths equal `lookback`/`horizon` — no future information leaks
  into the input window.
- Window counts (10 categories): lookback 7 → 480, 14 → 410, 28 → 270. The
  28-day lookback produces 0 training samples under the M3 split rules (see
  `docs/evaluation_protocol.md`), but the window file is still generated.
- Column contract: `sample_id, category_id, input_start_date, input_end_date,
  target_start_date, target_end_date, y`. Feature columns are added by the next
  step (`build_features`).
